<a href="https://colab.research.google.com/github/Lucky015-7/Telecom-Speech-to-Image-Conversion/blob/Frontend/Voice_to_Image_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Voice-to-Image Generation System
## SLT Mobitel Research Project - Google Colab Version

**ConnectPlus ISP - Intelligent Voice Understanding Platform**

---

### 📋 Instructions:
1. **Runtime → Change runtime type → GPU (T4)**
2. Run cells in order (Shift+Enter)
3. Upload your audio files when prompted
4. View generated images below

---

## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install diffusers transformers accelerate librosa soundfile scipy
!pip install openai-whisper
!pip install matplotlib pillow

print('✅ All packages installed successfully!')

## 🔧 Step 2: Import Libraries and Setup

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import ClapModel, ClapProcessor
from transformers import CLIPProcessor, CLIPModel
from diffusers import StableDiffusionPipeline
import librosa
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  Warning: GPU not available. Using CPU (will be slower)")

## 🎯 Step 3: Define Scene Vocabulary (66 Telecom Scenes)

In [ ]:
# ConnectPlus ISP Domain-Specific Scene Vocabulary
SCENE_VOCABULARY = [
    # Communication Infrastructure (10)
    "Damaged communication tower with engineers working",
    "Telecommunications tower against blue sky",
    "Engineers repairing cellular antennas on tower",
    "Fiber optic cable installation in progress",
    "Underground cable laying by technicians",
    "Satellite dish installation on building rooftop",
    "Network infrastructure maintenance crew",
    "Telecommunications equipment room with racks",
    "Cell tower with multiple antenna arrays",
    "Technician climbing communication tower",

    # Router & Home Network (10)
    "WiFi router with blinking orange warning light",
    "Modern router with green status LEDs",
    "Technician configuring home router setup",
    "Server room with network equipment racks",
    "Home office desk with router and modem",
    "Network switch with multiple ethernet cables",
    "Router installation in customer home",
    "Speed test running on laptop screen",
    "Mesh WiFi system in modern home",
    "Network diagnostic tools on computer screen",

    # Internet Devices (2)
    "Laptop showing browser connection error page",
    "Mobile phone displaying full WiFi signal strength",

    # Television Services (8)
    "Television screen showing no signal message",
    "Set-top box with connected cables",
    "Cable TV control room with monitors",
    "Technician installing satellite dish",
    "TV remote control and set-top box",
    "Digital TV signal quality test screen",
    "IPTV streaming interface on smart TV",
    "Broadcast equipment in control center",

    # Customer Service Operations (8)
    "Modern call center with rows of service agents",
    "Customer service representative with headset",
    "Service van parked outside residential house",
    "Field technician visiting customer home",
    "Help desk agent assisting customer",
    "Technical support team in office",
    "Mobile service unit in neighborhood",
    "Customer meeting with service representative",

    # Faults & Outages (8)
    "Network outage map with red affected areas",
    "Storm damage to telecommunications poles",
    "Power outage affecting city street at night",
    "Damaged cables after severe weather",
    "Emergency repair crew working at night",
    "Service interruption notification screen",
    "Broken utility pole with hanging wires",
    "Flooded telecommunications equipment room",

    # Billing & Plans (6)
    "Digital invoice displayed on computer screen",
    "Service plan comparison chart",
    "Contract document being signed",
    "Payment terminal for bill payment",
    "Mobile app showing billing details",
    "Customer reviewing service agreement",

    # Network & Data (8)
    "Global internet connectivity map visualization",
    "Modern data center with server racks",
    "Video streaming buffering symbol on screen",
    "Network traffic monitoring dashboard",
    "Cloud computing infrastructure diagram",
    "High-speed fiber optic network cables",
    "Network operations center with displays",
    "Internet speed test results on screen",

    # Environment & Context (6)
    "Suburban neighborhood with utility cables",
    "Person working from home office setup",
    "Network switch with illuminated LEDs",
    "Urban area with telecommunications infrastructure",
    "Residential area with cable connections",
    "Home internet setup in living room"
]

print(f"✅ Loaded {len(SCENE_VOCABULARY)} telecom service scenes")

## 🤖 Step 4: Load AI Models (Takes 5-10 minutes first time)

In [ ]:
print("Loading AI models... This may take 5-10 minutes on first run.\n")

# 1. Load Whisper (Speech Recognition)
print("📥 Loading Whisper ASR model...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-base")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base").to(device)
print("   ✅ Whisper loaded")

# 2. Load CLAP (Audio-Language Embedding)
print("📥 Loading CLAP audio embedding model...")
clap_processor = ClapProcessor.from_pretrained("laion/larger_clap_general")
clap_model = ClapModel.from_pretrained("laion/larger_clap_general").to(device)
print("   ✅ CLAP loaded")

# 3. Pre-encode scene vocabulary with CLAP
print("📥 Encoding scene vocabulary...")
inputs = clap_processor(text=SCENE_VOCABULARY, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    scene_output = clap_model.get_text_features(**inputs)
    # get_text_features returns a tensor directly for ClapModel — but older versions return an object
    # Handle both cases safely:
    if hasattr(scene_output, 'text_embeds'):
        scene_embeddings = scene_output.text_embeds
    elif hasattr(scene_output, 'pooler_output'):
        scene_embeddings = scene_output.pooler_output
    else:
        scene_embeddings = scene_output  # already a tensor
    scene_embeddings = scene_embeddings / scene_embeddings.norm(dim=-1, keepdim=True)
print("   ✅ Scene vocabulary encoded")

# 4. Load Stable Diffusion (Image Generation)
print("📥 Loading Stable Diffusion model...")
sd_pipeline = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)
if device == "cuda":
    sd_pipeline.enable_attention_slicing()
print("   ✅ Stable Diffusion loaded")

# 5. Load CLIP (Evaluation)
print("📥 Loading CLIP evaluation model...")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
print("   ✅ CLIP loaded")

print("\n🎉 All models loaded successfully!")

## 🎵 Step 5: Helper Functions

In [ ]:
def extract_acoustic_features(audio, sr):
    """Extract acoustic features from audio"""
    energy = np.mean(librosa.feature.rms(y=audio)[0])
    spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    brightness = np.mean(spectral_centroid) / sr
    zcr = np.mean(librosa.feature.zero_crossing_rate(audio)[0])
    duration = len(audio) / sr

    return {
        "energy": float(energy),
        "brightness": float(brightness),
        "zero_crossing_rate": float(zcr),
        "duration": float(duration)
    }

def compute_clip_score(image, text):
    """Compute CLIP similarity score"""
    inputs = clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
        image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
        similarity = (image_embeds @ text_embeds.T).item()
    return similarity

print("✅ Helper functions defined")

## 🔊 Step 6: Mode A - Audio-Direct Retrieval

In [ ]:
def process_mode_a(audio_path):
    """Mode A: Audio-Direct Retrieval using CLAP"""
    print("\n" + "="*60)
    print("🔊 MODE A: Audio-Direct Retrieval")
    print("="*60)

    # Load audio
    print("📥 Loading audio...")
    audio, sr = librosa.load(audio_path, sr=48000)

    # Extract features
    print("🎵 Extracting acoustic features...")
    features = extract_acoustic_features(audio, sr)
    print(f"   Duration: {features['duration']:.1f}s")
    print(f"   Energy: {features['energy']:.3f}")

    # Encode audio with CLAP
    print("🔊 Encoding audio with CLAP...")
    inputs = clap_processor(audios=audio, sampling_rate=sr, return_tensors="pt").to(device)
    with torch.no_grad():
        audio_embeds = clap_model.get_audio_features(**inputs)
        audio_embeds = audio_embeds / audio_embeds.norm(dim=-1, keepdim=True)

    # Find best matching scene
    print("🔍 Matching against scene vocabulary...")
    similarities = (audio_embeds @ scene_embeddings.T).cpu().numpy()[0]
    top_k_indices = np.argsort(similarities)[-5:][::-1]
    best_scene = SCENE_VOCABULARY[top_k_indices[0]]
    best_score = similarities[top_k_indices[0]]

    print(f"\n🎯 Best Match: {best_scene}")
    print(f"   Similarity Score: {best_score:.3f}")

    # Generate image
    print("\n🎨 Generating image with Stable Diffusion...")
    image = sd_pipeline(prompt=best_scene, num_inference_steps=50, guidance_scale=7.5).images[0]

    # Evaluate with CLIP
    print("📊 Evaluating with CLIP...")
    clip_score = compute_clip_score(image, best_scene)
    print(f"   CLIP Score: {clip_score:.3f}")

    # Display results
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(audio[:sr*5])  # First 5 seconds
    plt.title("Audio Waveform (First 5s)")
    plt.xlabel("Sample")
    plt.ylabel("Amplitude")

    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.title(f"Generated Image\nCLIP Score: {clip_score:.3f}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print("\n📋 Top 5 Matches:")
    for i, idx in enumerate(top_k_indices, 1):
        print(f"   {i}. {SCENE_VOCABULARY[idx]} ({similarities[idx]:.3f})")

    return image, best_scene, clip_score, features

print("✅ Mode A function ready")

## 🗣️ Step 7: Mode B - Speech Recognition + Text-Guided Generation

In [ ]:
def process_mode_b(audio_path):
    """Mode B: Speech Recognition + Text-Guided Generation"""
    print("\n" + "="*60)
    print("🗣️  MODE B: Speech Recognition + Text-Guided Generation")
    print("="*60)

    # Load audio
    print("📥 Loading audio...")
    audio, sr = librosa.load(audio_path, sr=48000)

    # Extract features
    print("🎵 Extracting acoustic features...")
    features = extract_acoustic_features(audio, sr)
    print(f"   Duration: {features['duration']:.1f}s")

    # Transcribe with Whisper
    print("🗣️  Transcribing with Whisper...")
    inputs = whisper_processor(audio, sampling_rate=sr, return_tensors="pt").to(device)
    with torch.no_grad():
        predicted_ids = whisper_model.generate(inputs.input_features)
        transcript = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    print(f"\n📝 Transcript: {transcript}")

    # Generate image
    print("\n🎨 Generating image with Stable Diffusion...")
    image = sd_pipeline(prompt=transcript, num_inference_steps=50, guidance_scale=7.5).images[0]

    # Evaluate with CLIP
    print("📊 Evaluating with CLIP...")
    clip_score = compute_clip_score(image, transcript)
    print(f"   CLIP Score: {clip_score:.3f}")

    # Display results
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(audio[:sr*5])  # First 5 seconds
    plt.title("Audio Waveform (First 5s)")
    plt.xlabel("Sample")
    plt.ylabel("Amplitude")

    plt.subplot(1, 2, 2)
    plt.imshow(image)
    plt.title(f"Generated Image\nCLIP Score: {clip_score:.3f}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    return image, transcript, clip_score, features

print("✅ Mode B function ready")

## 🎙️ Step 8: Upload Audio and Process

In [ ]:
print("📤 Upload your audio file (MP3, WAV, MPEG, etc.)\n")
uploaded = files.upload()

if uploaded:
    audio_filename = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {audio_filename}")
    print(f"   Size: {len(uploaded[audio_filename]) / 1024:.2f} KB")
else:
    print("❌ No file uploaded")

## 🔊 Step 9: Run Mode A (Audio-Direct)

In [ ]:
# Process with Mode A
print("🔊 Running Mode A: Audio-Direct Retrieval...")

# Load audio
audio, sr = librosa.load(audio_filename, sr=None)
print(f"   Loaded audio: duration={len(audio)/sr:.1f}s, sr={sr}Hz")

# Resample to 48kHz for CLAP
audio_48k = librosa.resample(audio, orig_sr=sr, target_sr=48000)

# Extract acoustic features
features_a = extract_acoustic_features(audio_48k, 48000)
print(f"   Duration: {features_a['duration']:.1f}s")
print(f"   Energy: {features_a['energy']:.3f}")

# Encode audio with CLAP
print("🔊 Encoding audio with CLAP...")
inputs_clap = clap_processor(
    audio=[audio_48k.tolist()],
    sampling_rate=48000,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    audio_output = clap_model.get_audio_features(**inputs_clap)
    if hasattr(audio_output, 'audio_embeds'):
        audio_embeds = audio_output.audio_embeds
    elif hasattr(audio_output, 'pooler_output'):
        audio_embeds = audio_output.pooler_output
    else:
        audio_embeds = audio_output
    audio_embeds = audio_embeds / audio_embeds.norm(dim=-1, keepdim=True)

# Match against scene vocabulary
print("🔍 Matching against scene vocabulary...")
similarities = (audio_embeds @ scene_embeddings.T).cpu().numpy()[0]
top_k_indices = np.argsort(similarities)[-5:][::-1]
best_scene = SCENE_VOCABULARY[top_k_indices[0]]
best_score = similarities[top_k_indices[0]]

print(f"\n🎯 Best Match: {best_scene}")
print(f"   Similarity Score: {best_score:.3f}")

# Generate image with Stable Diffusion
print("\n🎨 Generating image with Stable Diffusion...")
image_a = sd_pipeline(
    prompt=best_scene,
    num_inference_steps=50,
    guidance_scale=7.5
).images[0]

scene_a = best_scene

# Evaluate with CLIP
print("📊 Evaluating with CLIP...")
clip_a = compute_clip_score(image_a, best_scene)
print(f"   CLIP Score: {clip_a:.3f}")

# Display results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(audio_48k[:48000*5])
plt.title("Audio Waveform (First 5s)")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.subplot(1, 2, 2)
plt.imshow(image_a)
plt.title(f"Generated Image\nCLIP Score: {clip_a:.3f}")
plt.axis('off')
plt.tight_layout()
plt.show()

print("\n📋 Top 5 Matches:")
for i, idx in enumerate(top_k_indices, 1):
    print(f"   {i}. {SCENE_VOCABULARY[idx]} ({similarities[idx]:.3f})")

image_a.save('mode_a_output.png')
print("\n💾 Image saved as 'mode_a_output.png'")

## 🗣️ Step 10: Run Mode B (Speech-Guided)

In [ ]:
# Process with Mode B
print("🗣️ Running Mode B: Speech Recognition + Text-Guided Generation...")

# Load audio at original sr for Whisper
audio_b, sr_b = librosa.load(audio_filename, sr=16000)  # Whisper needs 16kHz
print(f"   Loaded audio: duration={len(audio_b)/sr_b:.1f}s")

# Extract acoustic features
features_b = extract_acoustic_features(audio_b, sr_b)
print(f"   Duration: {features_b['duration']:.1f}s")

# Transcribe with Whisper
print("🗣️ Transcribing with Whisper...")
whisper_inputs = whisper_processor(
    audio_b,
    sampling_rate=16000,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    predicted_ids = whisper_model.generate(whisper_inputs.input_features)
    transcript_b = whisper_processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]

print(f"\n📝 Transcript: {transcript_b}")

# Generate image with Stable Diffusion
print("\n🎨 Generating image with Stable Diffusion...")
image_b = sd_pipeline(
    prompt=transcript_b,
    num_inference_steps=50,
    guidance_scale=7.5
).images[0]

# Evaluate with CLIP
print("📊 Evaluating with CLIP...")
clip_b = compute_clip_score(image_b, transcript_b)
print(f"   CLIP Score: {clip_b:.3f}")

# Display results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(audio_b[:16000*5])
plt.title("Audio Waveform (First 5s)")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.subplot(1, 2, 2)
plt.imshow(image_b)
plt.title(f"Generated Image\nCLIP Score: {clip_b:.3f}")
plt.axis('off')
plt.tight_layout()
plt.show()

image_b.save('mode_b_output.png')
print("\n💾 Image saved as 'mode_b_output.png'")

## 📊 Step 11: Compare Results

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(image_a)
axes[0].set_title(f"Mode A: Audio-Direct\nCLIP Score: {clip_a:.3f}\n{scene_a[:50]}...", fontsize=10)
axes[0].axis('off')

axes[1].imshow(image_b)
axes[1].set_title(f"Mode B: Speech-Guided\nCLIP Score: {clip_b:.3f}\n{transcript_b[:50]}...", fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n📊 Comparison Summary:")
print(f"Mode A CLIP Score: {clip_a:.3f}")
print(f"Mode B CLIP Score: {clip_b:.3f}")
print(f"\nBetter Score: {'Mode A' if clip_a > clip_b else 'Mode B'}")

## 💾 Step 12: Download Results

In [ ]:
# Download generated images
print("📥 Download your generated images:")
files.download('mode_a_output.png')
files.download('mode_b_output.png')
print("\n✅ Downloads complete!")

---

## 🎉 Complete!

### 📋 Summary:
- ✅ **Mode A**: Audio-direct retrieval using CLAP
- ✅ **Mode B**: Speech recognition using Whisper + Stable Diffusion
- ✅ **Evaluation**: CLIP similarity scoring
- ✅ **Results**: Side-by-side comparison

### 🔄 To Process Another Audio:
Run cells 8-12 again with a new audio file.

---

**SLT Mobitel Research Project**  
*ConnectPlus ISP - Intelligent Voice Understanding Platform*  
*Version 3.0 - 2026*